In [0]:
%sql
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.bronze.stores') AS
SELECT 
  *,
  _metadata.file_path AS _source_file_path,
  current_timestamp() AS ingested_at
FROM read_files(
  '/Volumes/' || :catalog || '/landing/raw/*/stores/',
  format => 'csv',
  header => true,
  inferSchema => false
);

In [0]:
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.bronze.customers') AS
SELECT * FROM read_files(
  '/Volumes/' || :catalog || '/landing/raw/*/customers/',
  format => 'csv',
  header => true,
  inferSchema => false
) WHERE 1=0;

In [0]:
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.bronze.customers') AS
SELECT
  *,
  _metadata.file_path AS _source_file_path,
  current_timestamp() AS ingested_at
FROM read_files(
  '/Volumes/' || :catalog || '/landing/raw/*/customers/',
  format => 'csv',
  header => true,
  inferSchema => false
);

In [0]:
select * from fk_CE_2026_07_1372.bronze.customers

In [0]:
CREATE or REPLACE TABLE IDENTIFIER(:catalog || '.bronze.products') as SELECT * FROM read_files(
  '/Volumes/fk_ce_2026_07_1372/landing/raw/*/products/',
  format => 'json',
  multiLine => true  
);


In [0]:
%python
from pyspark.sql.functions import col, current_timestamp

catalog = dbutils.widgets.get('catalog')
checkpoint_path = f'/Volumes/{catalog}/ops/pipeline_state/orders'
schema_path = f'/Volumes/{catalog}/ops/pipeline_state/orders_schema'
source_path = f'/Volumes/{catalog}/landing/raw/*/orders/'
target_table = f'{catalog}.bronze.orders'

query = (spark.readStream
  .format('cloudFiles')
  .option('cloudFiles.format', 'json')
  .option('cloudFiles.schemaLocation', schema_path)
  .option('cloudFiles.schemaEvolutionMode', 'addNewColumns')
  .option('rescuedDataColumn', '_rescued_data')
  .load(source_path)
  .withColumn('_source_file_path', col('_metadata')['file_path'])
  .withColumn('ingested_at', current_timestamp())
  .writeStream
  .trigger(availableNow=True)
  .option('checkpointLocation', checkpoint_path)
  .option('mergeSchema', 'true')
  .toTable(target_table))

query.awaitTermination()
print(f'Orders loaded to {target_table}')